# H3: Joint Fitness Model Comparison

M4 (joint W with ω + κ) vs M1 (effort-only), M2 (threat-only), M3 (single-parameter).

All models compared on joint (choice + vigor) WAIC and PSIS-LOO.
Results read from `results/stats/joint_optimal/mcmc_model_comparison.csv`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
from pathlib import Path

REPO_ROOT = Path(os.getcwd())
for _ in range(5):
    if (REPO_ROOT / '.git').exists(): break
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

# Load MCMC results
mc = pd.read_csv('results/stats/joint_optimal/mcmc_model_comparison.csv')
mc = mc.set_index('Model')

def get(model, col, default=np.nan):
    if model in mc.index and col in mc.columns:
        return mc.loc[model, col]
    return default

# Extract key metrics for preregistered models
models = {}
for m in ['M1', 'M2', 'M3', 'M4']:
    models[m] = {
        'waic': get(m, 'WAIC'), 'loo': get(m, 'LOO'),
        'acc': get(m, 'choice_acc'), 'vig': get(m, 'vigor_r2'),
        'desc': get(m, 'Description', m),
    }

m4 = models['M4']
print('MCMC Model Comparison (Preregistered)')
print('=' * 60)
for name, v in models.items():
    vig_str = f", vig r²={v['vig']:.3f}" if not np.isnan(v['vig']) else ""
    loo_str = f", LOO={v['loo']:.0f}" if not np.isnan(v['loo']) else ""
    print(f"  {name} ({v['desc']}): WAIC={v['waic']:.0f}{loo_str}, acc={v['acc']:.3f}{vig_str}")

## H3a: M4 vs M1 — Threat matters beyond effort

In [ ]:
dw = models['M1']['waic'] - m4['waic']
dl = models['M1']['loo'] - m4['loo'] if not np.isnan(models['M1']['loo']) else np.nan
print('H3a — M4 vs M1 (Effort-only)')
print('=' * 50)
print(f"  ΔWAIC = {dw:+.0f}")
if not np.isnan(dl): print(f"  ΔLOO  = {dl:+.0f}")
agree = dw > 0 and (np.isnan(dl) or dl > 0)
print(f"  H3a: {'PASS ✓' if agree else 'FAIL ✗'}")

## H3b: M4 vs M2 — Individual effort differences matter

In [ ]:
dw = models['M2']['waic'] - m4['waic']
dl = models['M2']['loo'] - m4['loo'] if not np.isnan(models['M2']['loo']) else np.nan
print('H3b — M4 vs M2 (Threat-only)')
print('=' * 50)
print(f"  ΔWAIC = {dw:+.0f}")
if not np.isnan(dl): print(f"  ΔLOO  = {dl:+.0f}")
agree = dw > 0 and (np.isnan(dl) or dl > 0)
print(f"  H3b: {'PASS ✓' if agree else 'FAIL ✗'}")

## H3c: M4 vs M3 — Capture cost and effort cost are separable

In [ ]:
dw = models['M3']['waic'] - m4['waic']
dl = models['M3']['loo'] - m4['loo'] if not np.isnan(models['M3']['loo']) else np.nan
print('H3c — M4 vs M3 (Single-parameter)')
print('=' * 50)
print(f"  ΔWAIC = {dw:+.0f}")
if not np.isnan(dl): print(f"  ΔLOO  = {dl:+.0f}")
agree = dw > 0 and (np.isnan(dl) or dl > 0)
print(f"  H3c: {'PASS ✓' if agree else 'FAIL ✗'}")

## Summary

In [ ]:
print('H3 SUMMARY (MCMC)')
print('=' * 60)
tests = []
for alt, h in [('M1', 'H3a'), ('M2', 'H3b'), ('M3', 'H3c')]:
    dw = models[alt]['waic'] - m4['waic']
    dl = models[alt]['loo'] - m4['loo'] if not np.isnan(models[alt]['loo']) else np.nan
    waic_ok = dw > 0
    loo_ok = np.isnan(dl) or dl > 0
    tests.append((h, alt, dw, dl, waic_ok and loo_ok))

print(f"{'Test':<8} {'Alt':<15} {'ΔWAIC':>8} {'ΔLOO':>8} {'Pass':>6}")
print('-' * 48)
for h, alt, dw, dl, p in tests:
    dl_str = f'{dl:>+8.0f}' if not np.isnan(dl) else '     N/A'
    print(f"{h:<8} {alt:<15} {dw:>+8.0f} {dl_str} {'✓':>6}" if p else f"{h:<8} {alt:<15} {dw:>+8.0f} {dl_str} {'✗':>6}")
n_pass = sum(1 for *_, p in tests if p)
print(f'\n{n_pass}/3 tests pass.')